# Benchmarking Datasets Analysis

This notebook analyzes each dataset in the benchmarking datasets folder:
1. Computes predictor correlations (Pearson, Spearman, and Mutual Information)
2. Runs leave-one-out analysis to identify important predictors
3. Creates 60/40 train/test splits for donor/receiver data
4. Runs autoimpute using the donor/receiver splits

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from microimpute import (
    method_comparison_results,
    compute_predictor_correlations,
    leave_one_out_analysis,
    autoimpute,
    QRF,
    QuantReg, 
    OLS, 
    Matching,
)
from microimpute.comparisons import compare_distributions

# Import benchmark utilities for CIA analysis and cross-dataset summary
from benchmark_utils import (
    progressive_predictor_exclusion,
    plot_cia_degradation_curves,
    create_benchmark_summary_table,
    create_benchmark_heatmap,
    run_cia_analysis_for_dataset,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

In [2]:
DATASETS_DIR = "datasets"

summary_df = pd.read_csv(os.path.join(DATASETS_DIR, "download_summary.csv"))
print("Available datasets:")
print(summary_df[["name", "target", "n_rows", "n_cols"]])

Available datasets:
                   name                 target  n_rows  n_cols
0              space_ga          ln(VOTES/POP)    3107       7
1          wine_quality                quality    6497      12
2             elevators                   Goal   16599      19
3      Brazilian_houses            total_(BRL)   10692      13
4  OnlineNewsPopularity                 shares   39644      60
5               abalone  Class_number_of_rings    4177       9
6           house_sales                  price   21613      22


In [3]:
def load_dataset(name):
    """Load a dataset by name and return the DataFrame."""
    # Clean name to match file naming convention
    clean_name = name.lower().replace(" ", "_").replace("-", "_")
    file_path = os.path.join(DATASETS_DIR, f"{clean_name}.csv")
    return pd.read_csv(file_path)

---
## 1. Space GA Dataset

Georgia county voting data with geographic variables.

In [4]:
df_space_ga = load_dataset("space_ga")
print(f"\nShape: {df_space_ga.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df_space_ga.columns):
    print(f"  {i+1}. {col}")
print(f"\nData types:")
print(df_space_ga.dtypes)
print(f"\nFirst few rows:")
df_space_ga.head()


Shape: (3107, 7)

Column names:
  1. POP
  2. EDUCATION
  3. HOUSES
  4. INCOME
  5. XCOORD
  6. YCOORD
  7. ln(VOTES/POP)

Data types:
POP              float64
EDUCATION        float64
HOUSES           float64
INCOME           float64
XCOORD           float64
YCOORD           float64
ln(VOTES/POP)    float64
dtype: object

First few rows:


,POP,EDUCATION,HOUSES,INCOME,XCOORD,YCOORD,ln(VOTES/POP)
0,9.972920,9.246286,9.004054,12.134915,-86641472.0,32542207.0,-0.661559
1,10.903347,10.221214,9.965758,13.056638,-87754736.0,30654881.0,-0.650859
2,9.722205,8.753529,8.707648,11.630628,-85388993.0,31863073.0,-0.617114
3,9.273691,8.183118,8.277412,11.243712,-87126855.0,32996943.0,-0.639070
4,10.151519,9.207737,9.240676,12.155100,-86566214.0,33979740.0,-0.700274


In [5]:
space_ga_target = "ln(VOTES/POP)"  # Target variable to impute
space_ga_predictors = [col for col in df_space_ga.columns if col != space_ga_target]

print(f"Target variable: {space_ga_target}")
print(f"Predictors ({len(space_ga_predictors)}): {space_ga_predictors}")

Target variable: ln(VOTES/POP)
Predictors (6): ['POP', 'EDUCATION', 'HOUSES', 'INCOME', 'XCOORD', 'YCOORD']


In [6]:
print("Computing predictor correlations...")
space_ga_correlations = compute_predictor_correlations(
    data=df_space_ga,
    predictors=space_ga_predictors,
    imputed_variables=[space_ga_target],
    method="all"
)

print("\n--- Pearson Correlation Matrix ---")
display(space_ga_correlations["pearson"])

print("\n--- Spearman Correlation Matrix ---")
display(space_ga_correlations["spearman"])

print("\n--- Mutual Information Matrix ---")
display(space_ga_correlations["mutual_info"])

print("\n--- Predictor-Target Mutual Information ---")
display(space_ga_correlations["predictor_target_mi"].sort_values(
    by=space_ga_target, ascending=False
))

Computing predictor correlations...

--- Pearson Correlation Matrix ---


,POP,EDUCATION,HOUSES,INCOME,XCOORD,YCOORD
POP,1.000000,0.987472,0.994002,0.992899,0.267719,-0.040601
EDUCATION,0.987472,1.000000,0.984365,0.993133,0.199129,0.047424
HOUSES,0.994002,0.984365,1.000000,0.988084,0.268707,-0.030783
INCOME,0.992899,0.993133,0.988084,1.000000,0.231344,-0.008157
XCOORD,0.267719,0.199129,0.268707,0.231344,1.000000,-0.193174
YCOORD,-0.040601,0.047424,-0.030783,-0.008157,-0.193174,1.000000



--- Spearman Correlation Matrix ---


,POP,EDUCATION,HOUSES,INCOME,XCOORD,YCOORD
POP,1.000000,0.982208,0.994249,0.990724,0.353281,-0.022140
EDUCATION,0.982208,1.000000,0.980917,0.991055,0.280971,0.082580
HOUSES,0.994249,0.980917,1.000000,0.987013,0.350405,-0.010133
INCOME,0.990724,0.991055,0.987013,1.000000,0.316362,0.020701
XCOORD,0.353281,0.280971,0.350405,0.316362,1.000000,-0.139110
YCOORD,-0.022140,0.082580,-0.010133,0.020701,-0.139110,1.000000



--- Mutual Information Matrix ---


,POP,EDUCATION,HOUSES,INCOME,XCOORD,YCOORD
POP,1.000000,0.173367,0.229931,0.187904,0.018096,0.004794
EDUCATION,0.173367,1.000000,0.167237,0.195508,0.016309,0.004089
HOUSES,0.229931,0.167237,1.000000,0.178265,0.019829,0.006102
INCOME,0.187904,0.195508,0.178265,1.000000,0.015291,0.006218
XCOORD,0.018096,0.016309,0.019829,0.015291,1.000000,0.000000
YCOORD,0.004794,0.004089,0.006102,0.006218,0.000000,1.000000



--- Predictor-Target Mutual Information ---


,ln(VOTES/POP)
YCOORD,0.021069
XCOORD,0.017490
POP,0.012771
HOUSES,0.011503
INCOME,0.010931
EDUCATION,0.009178


In [7]:
print("Running leave-one-out analysis...")
space_ga_loo = leave_one_out_analysis(
    data=df_space_ga,
    predictors=space_ga_predictors,
    imputed_variables=[space_ga_target],
    model_class=QRF,
    train_size=0.6,
    n_jobs=1
)

print("\n--- Leave-One-Out Results ---")
display(space_ga_loo)

Running leave-one-out analysis...


Leave-one-out analysis:   0%|          | 0/6 [00:00<?, ?it/s]


--- Leave-One-Out Results ---


,predictor_removed,avg_quantile_loss,avg_log_loss,loss_increase,relative_impact,baseline_quantile_loss,baseline_log_loss
5,YCOORD,0.047892,0,0.008613,21.927196,0.039279,0
4,XCOORD,0.045017,0,0.005738,14.608486,0.039279,0
2,HOUSES,0.039934,0,0.000655,1.666995,0.039279,0
0,POP,0.039710,0,0.000431,1.096066,0.039279,0
1,EDUCATION,0.039361,0,0.000082,0.208418,0.039279,0
3,INCOME,0.039025,0,-0.000254,-0.647140,0.039279,0


In [8]:
print("Creating 60/40 donor/receiver split...")
space_ga_donor, space_ga_receiver = train_test_split(
    df_space_ga, train_size=0.6, random_state=42
)

# Remove target from receiver (simulating missing data)
space_ga_receiver_no_target = space_ga_receiver.drop(columns=[space_ga_target])

print(f"Donor size: {len(space_ga_donor)}")
print(f"Receiver size: {len(space_ga_receiver)}")

print("\nRunning autoimpute...")
space_ga_result = autoimpute(
    donor_data=space_ga_donor,
    receiver_data=space_ga_receiver_no_target.copy(),
    predictors=space_ga_predictors,
    imputed_variables=[space_ga_target],
    impute_all=True,
    log_level="INFO"
)

print("\n--- Autoimpute CV Benchmarking Results ---")
comparison_viz = method_comparison_results(
    data=space_ga_result.cv_results,
    metric="quantile_loss",
    data_format="wide",
)
fig = comparison_viz.plot(
    title="Space GA Dataset Benchmarking Results",
    show_mean=True,
)
fig.show()

Creating 60/40 donor/receiver split...
Donor size: 1864
Receiver size: 1243

Running autoimpute...


AutoImputation progress:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    9.1s remaining:   13.7s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    9.1s remaining:    6.1s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    9.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.19906830787658691s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    0.2s remaining:    0.3s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    0.2s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.05776691436767578s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    0.1s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done   3 o


--- Autoimpute CV Benchmarking Results ---


In [9]:
print("\n--- Distribution Comparison: Imputed vs Ground Truth (Space GA) ---")
# Store Wasserstein distances for all methods
space_ga_wasserstein = {}

distribution_comparison_space_ga = compare_distributions(
    donor_data=space_ga_receiver,  # Ground truth test split
    receiver_data=space_ga_result.receiver_data,  # Imputed values
    imputed_variables=[space_ga_target],
)
best_method_name = space_ga_result.fitted_models["best_method"].__class__.__name__ 
best_method_name = best_method_name.replace("Results", "")
space_ga_wasserstein[best_method_name] = distribution_comparison_space_ga[
    distribution_comparison_space_ga['Metric'] == 'wasserstein_distance'
]['Distance'].values[0]
print(f"{best_method_name}: Wasserstein Distance = {space_ga_wasserstein.get(best_method_name, 'N/A'):.4f}")
            
for method_name, imputations in space_ga_result.imputations.items():
    # Skip the 'best_method' (it's a duplicate)
    if method_name == 'best_method':              
        continue
                                                                            
    # Create a copy of receiver data with this method's imputations 
    receiver_with_imputations = space_ga_receiver_no_target.copy() 
                                                    
    # Handle both dict (quantile->DataFrame) and DataFrame formats 
    if isinstance(imputations, dict):                   
        # Get median quantile (0.5) imputations                   
        imp_df = imputations.get(0.5, list(imputations.values())[0]) 
    else:                                                   
        imp_df = imputations
                                   
    # Add imputed values                                    
    for var in [space_ga_target]:
        if var in imp_df.columns:                                    
            receiver_with_imputations[var] = imp_df[var].values    
                                                
    # Calculate distribution comparison                 
    dist_comparison = compare_distributions( 
        donor_data=space_ga_receiver,  # Ground truth                       
        receiver_data=receiver_with_imputations,
        imputed_variables=[space_ga_target],                       
    )

    # Extract Wasserstein distance 
    wd = dist_comparison[dist_comparison['Metric'] == 'wasserstein_distance']['Distance'].values                  
    space_ga_wasserstein[method_name] = wd[0]
                                
    print(f"{method_name}: Wasserstein Distance = {space_ga_wasserstein.get(method_name, 'N/A'):.4f}")


--- Distribution Comparison: Imputed vs Ground Truth (Space GA) ---
OLS: Wasserstein Distance = 0.0272
QRF: Wasserstein Distance = 0.0101
QuantReg: Wasserstein Distance = 0.0981
Matching: Wasserstein Distance = 0.0111
MDN: Wasserstein Distance = 0.0504


### CIA Sensitivity Analysis

Measuring sensitivity to the Conditional Independence Assumption by progressively removing predictors.

In [10]:
print("Running CIA sensitivity analysis...")
space_ga_cia_results = run_cia_analysis_for_dataset(
    data=df_space_ga,
    predictors=space_ga_predictors,
    imputed_variables=[space_ga_target],
    ordering="mutual_info",
)

# Plot degradation curves
space_ga_cia_fig = plot_cia_degradation_curves(
    space_ga_cia_results,
    title="CIA Sensitivity: Space Ga"
)
space_ga_cia_fig.show()

# Display sensitivity scores
print("\nSensitivity Scores (higher = more sensitive to predictor removal):")
for method, results in space_ga_cia_results.items():
    print(f"  {method}: {results['sensitivity_score']:.3f}")

Running CIA sensitivity analysis...


Progressive exclusion:   0%|          | 0/6 [00:00<?, ?it/s]

  QRF sensitivity score: 1.443


Progressive exclusion:   0%|          | 0/6 [00:00<?, ?it/s]

  OLS sensitivity score: 1.192


...

Progressive exclusion:   0%|          | 0/6 [00:00<?, ?it/s]

  Matching sensitivity score: 1.267


Progressive exclusion:   0%|          | 0/6 [00:00<?, ?it/s]

  MDN sensitivity score: 1.312



Sensitivity Scores (higher = more sensitive to predictor removal):
  QRF: 1.443
  OLS: 1.192
  QuantReg: 0.688
  Matching: 1.267
  MDN: 1.312


---
## 2. Elevators Dataset

Aircraft control system data from an F-16 simulation.

In [11]:
df_elevators = load_dataset("elevators")
print(f"\nShape: {df_elevators.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df_elevators.columns):
    print(f"  {i+1}. {col}")
print(f"\nData types:")
print(df_elevators.dtypes)
print(f"\nFirst few rows:")
df_elevators.head()


Shape: (16599, 19)

Column names:
  1. climbRate
  2. Sgz
  3. p
  4. q
  5. curRoll
  6. absRoll
  7. diffClb
  8. diffRollRate
  9. diffDiffClb
  10. SaTime1
  11. SaTime2
  12. SaTime3
  13. SaTime4
  14. diffSaTime1
  15. diffSaTime2
  16. diffSaTime3
  17. diffSaTime4
  18. Sa
  19. Goal

Data types:
climbRate       float64
Sgz             float64
p               float64
q               float64
curRoll         float64
absRoll         float64
diffClb         float64
diffRollRate    float64
diffDiffClb     float64
SaTime1         float64
SaTime2         float64
SaTime3         float64
SaTime4         float64
diffSaTime1     float64
diffSaTime2     float64
diffSaTime3     float64
diffSaTime4     float64
Sa              float64
Goal            float64
dtype: object

First few rows:


,climbRate,Sgz,p,q,curRoll,absRoll,diffClb,diffRollRate,diffDiffClb,SaTime1,SaTime2,SaTime3,SaTime4,diffSaTime1,diffSaTime2,diffSaTime3,diffSaTime4,Sa,Goal
0,118.0,-55.0,-0.28,-0.08,-0.2,-11.0,11.0,0.005,-0.2,-0.0010,-0.0010,-0.0010,-0.0010,0.0000,0.0,0.0,0.0,-0.0010,0.031
1,390.0,-45.0,-0.06,-0.07,-0.6,-12.0,11.0,0.010,-0.2,-0.0008,-0.0008,-0.0008,-0.0008,0.0000,0.0,0.0,0.0,-0.0008,0.034
2,68.0,6.0,0.11,0.15,0.6,-10.0,-9.0,-0.003,-0.2,-0.0011,-0.0010,-0.0010,-0.0010,-0.0002,0.0,0.0,0.0,-0.0010,0.033
3,-358.0,-12.0,-0.20,0.13,-0.3,-11.0,-7.0,0.001,-0.1,-0.0010,-0.0010,-0.0010,-0.0010,0.0000,0.0,0.0,0.0,-0.0010,0.032
4,-411.0,-19.0,-0.18,0.02,-0.5,-11.0,-3.0,0.002,1.2,-0.0010,-0.0010,-0.0010,-0.0010,0.0000,0.0,0.0,0.0,-0.0010,0.030


In [12]:
elevators_target = "Goal"  # Target variable to impute
elevators_predictors = ["climbRate", "Sgz", "curRoll", "absRoll", "SaTime1", "SaTime2", "SaTime3", "SaTime4"]

print(f"Target variable: {elevators_target}")
print(f"Predictors ({len(elevators_predictors)}): {elevators_predictors}")

Target variable: Goal
Predictors (8): ['climbRate', 'Sgz', 'curRoll', 'absRoll', 'SaTime1', 'SaTime2', 'SaTime3', 'SaTime4']


In [13]:
print("Computing predictor correlations...")
elevators_correlations = compute_predictor_correlations(
    data=df_elevators,
    predictors=elevators_predictors,
    imputed_variables=[elevators_target],
    method="all"
)

print("\n--- Pearson Correlation Matrix ---")
display(elevators_correlations["pearson"])

print("\n--- Spearman Correlation Matrix ---")
display(elevators_correlations["spearman"])

print("\n--- Mutual Information Matrix ---")
display(elevators_correlations["mutual_info"])

print("\n--- Predictor-Target Mutual Information ---")
display(elevators_correlations["predictor_target_mi"].sort_values(
    by=elevators_target, ascending=False
))

Computing predictor correlations...

--- Pearson Correlation Matrix ---


,climbRate,Sgz,curRoll,absRoll,SaTime1,SaTime2,SaTime3,SaTime4
climbRate,1.000000,-0.013900,-0.146812,-0.023482,0.094790,0.104038,0.104072,0.107007
Sgz,-0.013900,1.000000,0.017043,0.077119,0.062284,0.070309,0.070308,0.076198
curRoll,-0.146812,0.017043,1.000000,0.194465,-0.230882,-0.227539,-0.227519,-0.224023
absRoll,-0.023482,0.077119,0.194465,1.000000,0.683132,0.678935,0.678954,0.678385
SaTime1,0.094790,0.062284,-0.230882,0.683132,1.000000,0.988001,0.987995,0.976332
SaTime2,0.104038,0.070309,-0.227539,0.678935,0.988001,1.000000,0.999991,0.994399
SaTime3,0.104072,0.070308,-0.227519,0.678954,0.987995,0.999991,1.000000,0.994408
SaTime4,0.107007,0.076198,-0.224023,0.678385,0.976332,0.994399,0.994408,1.000000



--- Spearman Correlation Matrix ---


,climbRate,Sgz,curRoll,absRoll,SaTime1,SaTime2,SaTime3,SaTime4
climbRate,1.000000,-0.017436,-0.147546,-0.019983,0.118637,0.127793,0.127798,0.131181
Sgz,-0.017436,1.000000,0.014349,0.088700,0.056970,0.065403,0.065405,0.071326
curRoll,-0.147546,0.014349,1.000000,0.196730,-0.257374,-0.252452,-0.252451,-0.247426
absRoll,-0.019983,0.088700,0.196730,1.000000,0.706120,0.700693,0.700695,0.699823
SaTime1,0.118637,0.056970,-0.257374,0.706120,1.000000,0.986084,0.986084,0.974161
SaTime2,0.127793,0.065403,-0.252452,0.700693,0.986084,1.000000,1.000000,0.993382
SaTime3,0.127798,0.065405,-0.252451,0.700695,0.986084,1.000000,1.000000,0.993382
SaTime4,0.131181,0.071326,-0.247426,0.699823,0.974161,0.993382,0.993382,1.000000



--- Mutual Information Matrix ---


,climbRate,Sgz,curRoll,absRoll,SaTime1,SaTime2,SaTime3,SaTime4
climbRate,1.000000,0.001757,0.001254,0.002738,0.001881,0.004853,0.004776,0.006018
Sgz,0.001757,1.000000,0.000000,0.004338,0.003036,0.003655,0.003656,0.004861
curRoll,0.001254,0.000000,1.000000,0.053293,0.013725,0.013256,0.013214,0.012820
absRoll,0.002738,0.004338,0.053293,1.000000,0.095271,0.093617,0.093553,0.094038
SaTime1,0.001881,0.003036,0.013725,0.095271,1.000000,0.505716,0.505646,0.458947
SaTime2,0.004853,0.003655,0.013256,0.093617,0.505716,1.000000,0.691706,0.584177
SaTime3,0.004776,0.003656,0.013214,0.093553,0.505646,0.691706,1.000000,0.584307
SaTime4,0.006018,0.004861,0.012820,0.094038,0.458947,0.584177,0.584307,1.000000



--- Predictor-Target Mutual Information ---


,Goal
SaTime1,0.055697
SaTime3,0.048387
SaTime2,0.048359
SaTime4,0.047626
absRoll,0.025873
climbRate,0.013537
Sgz,0.005171
curRoll,0.000220


In [14]:
print("Running leave-one-out analysis...")
elevators_loo = leave_one_out_analysis(
    data=df_elevators,
    predictors=elevators_predictors,
    imputed_variables=[elevators_target],
    model_class=QRF,
    train_size=0.6,
    n_jobs=1
)

print("\n--- Leave-One-Out Results ---")
display(elevators_loo)

Running leave-one-out analysis...


Leave-one-out analysis:   0%|          | 0/8 [00:00<?, ?it/s]


--- Leave-One-Out Results ---


,predictor_removed,avg_quantile_loss,avg_log_loss,loss_increase,relative_impact,baseline_quantile_loss,baseline_log_loss
3,absRoll,0.001463,0,1.562464e-04,11.955247,0.001307,0
2,curRoll,0.001458,0,1.509115e-04,11.547046,0.001307,0
0,climbRate,0.001414,0,1.066808e-04,8.162716,0.001307,0
1,Sgz,0.001355,0,4.820981e-05,3.688790,0.001307,0
4,SaTime1,0.001331,0,2.440591e-05,1.867426,0.001307,0
7,SaTime4,0.001314,0,7.272511e-06,0.556459,0.001307,0
6,SaTime3,0.001307,0,-6.777108e-08,-0.005186,0.001307,0
5,SaTime2,0.001306,0,-5.084813e-07,-0.038907,0.001307,0


In [15]:
print("Creating 60/40 donor/receiver split...")
elevators_donor, elevators_receiver = train_test_split(
    df_elevators, train_size=0.6, random_state=42
)

elevators_receiver_no_target = elevators_receiver.drop(columns=[elevators_target])

print(f"Donor size: {len(elevators_donor)}")
print(f"Receiver size: {len(elevators_receiver)}")

print("\nRunning autoimpute...")
elevators_result = autoimpute(
    donor_data=elevators_donor,
    receiver_data=elevators_receiver_no_target.copy(),
    predictors=elevators_predictors,
    imputed_variables=[elevators_target],
    impute_all=True,
    log_level="INFO"
)

print("\n--- Autoimpute CV Results ---")
comparison_viz = method_comparison_results(
    data=elevators_result.cv_results,
    metric="quantile_loss",
)
fig = comparison_viz.plot(
    title="Elevators Dataset Benchmarking Results",
    show_mean=True,
)
fig.show()

Creating 60/40 donor/receiver split...
Donor size: 9959
Receiver size: 6640

Running autoimpute...


AutoImputation progress:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    8.2s remaining:   12.3s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    8.2s remaining:    5.5s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    8.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    0.4s remaining:    0.6s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    0.4s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.6s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    1.6s remaining:    2.4s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    1.7s remaining:    1.2s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    2.0s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    8.6s
[Parallel(n_


--- Autoimpute CV Results ---


In [16]:
elevators_result.cv_results.pop('MDN', None)

print("\n--- Autoimpute CV Results (Zoomed In Without MDN) ---")
comparison_viz = method_comparison_results(
    data=elevators_result.cv_results,
    metric="quantile_loss",
)
fig = comparison_viz.plot(
    title="Elevators Dataset Benchmarking Results",
    show_mean=True,
)
fig.show()


--- Autoimpute CV Results (Zoomed In Without MDN) ---


In [17]:
print("\n--- Distribution Comparison: Imputed vs Ground Truth (Elevators) ---")
# Store Wasserstein distances for all methods
elevators_wasserstein = {}

distribution_comparison_elevators = compare_distributions(
    donor_data=elevators_receiver,  # Ground truth test split
    receiver_data=elevators_result.receiver_data,  # Imputed values
    imputed_variables=[elevators_target],
)
best_method_name = elevators_result.fitted_models["best_method"].__class__.__name__ 
best_method_name = best_method_name.replace("Results", "")
elevators_wasserstein[best_method_name] = distribution_comparison_elevators[
    distribution_comparison_elevators['Metric'] == 'wasserstein_distance'
]['Distance'].values[0]
print(f"{best_method_name}: Wasserstein Distance = {elevators_wasserstein.get(best_method_name, 'N/A'):.4f}")
            
for method_name, imputations in elevators_result.imputations.items():
    # Skip the 'best_method' (it's a duplicate)
    if method_name == 'best_method':              
        continue
                                                                            
    # Create a copy of receiver data with this method's imputations 
    receiver_with_imputations = elevators_receiver_no_target.copy() 
                                                    
    # Handle both dict (quantile->DataFrame) and DataFrame formats 
    if isinstance(imputations, dict):                   
        # Get median quantile (0.5) imputations                   
        imp_df = imputations.get(0.5, list(imputations.values())[0]) 
    else:                                                   
        imp_df = imputations
                                   
    # Add imputed values                                    
    for var in [elevators_target]:
        if var in imp_df.columns:                                    
            receiver_with_imputations[var] = imp_df[var].values    
                                                
    # Calculate distribution comparison                 
    dist_comparison = compare_distributions( 
        donor_data=elevators_receiver,  # Ground truth                       
        receiver_data=receiver_with_imputations,
        imputed_variables=[elevators_target],                       
    )

    # Extract Wasserstein distance 
    wd = dist_comparison[dist_comparison['Metric'] == 'wasserstein_distance']['Distance'].values                  
    elevators_wasserstein[method_name] = wd[0]
                                
    print(f"{method_name}: Wasserstein Distance = {elevators_wasserstein.get(method_name, 'N/A'):.4f}")


--- Distribution Comparison: Imputed vs Ground Truth (Elevators) ---
QuantReg: Wasserstein Distance = 0.0015
QRF: Wasserstein Distance = 0.0003
OLS: Wasserstein Distance = 0.0014
Matching: Wasserstein Distance = 0.0002
MDN: Wasserstein Distance = 0.0200


### CIA Sensitivity Analysis

Measuring sensitivity to the Conditional Independence Assumption by progressively removing predictors.

In [18]:
print("Running CIA sensitivity analysis...")
elevators_cia_results = run_cia_analysis_for_dataset(
    data=df_elevators,
    predictors=elevators_predictors,
    imputed_variables=[elevators_target],
    ordering="mutual_info",
)

# Plot degradation curves
elevators_cia_fig = plot_cia_degradation_curves(
    elevators_cia_results,
    title="CIA Sensitivity: Elevators"
)
elevators_cia_fig.show()

# Display sensitivity scores
print("\nSensitivity Scores (higher = more sensitive to predictor removal):")
for method, results in elevators_cia_results.items():
    print(f"  {method}: {results['sensitivity_score']:.3f}")

Running CIA sensitivity analysis...


Progressive exclusion:   0%|          | 0/8 [00:00<?, ?it/s]

  QRF sensitivity score: 1.293


Progressive exclusion:   0%|          | 0/8 [00:00<?, ?it/s]

  OLS sensitivity score: 1.180


...

Progressive exclusion:   0%|          | 0/8 [00:00<?, ?it/s]

  Matching sensitivity score: 1.050


Progressive exclusion:   0%|          | 0/8 [00:00<?, ?it/s]

  MDN sensitivity score: 2.055



Sensitivity Scores (higher = more sensitive to predictor removal):
  QRF: 1.293
  OLS: 1.180
  QuantReg: 1.174
  Matching: 1.050
  MDN: 2.055


---
## 3. Brazilian Houses Dataset

Real estate data from Brazil with property characteristics and prices.

In [19]:
df_brazilian = load_dataset("brazilian_houses")
print(f"\nShape: {df_brazilian.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df_brazilian.columns):
    print(f"  {i+1}. {col}")
print(f"\nData types:")
print(df_brazilian.dtypes)
print(f"\nFirst few rows:")
df_brazilian.head()


Shape: (10692, 13)

Column names:
  1. city
  2. area
  3. rooms
  4. bathroom
  5. parking_spaces
  6. floor
  7. animal
  8. furniture
  9. hoa_(BRL)
  10. rent_amount_(BRL)
  11. property_tax_(BRL)
  12. fire_insurance_(BRL)
  13. total_(BRL)

Data types:
city                    object
area                     int64
rooms                    int64
bathroom                 int64
parking_spaces           int64
floor                    int64
animal                  object
furniture               object
hoa_(BRL)                int64
rent_amount_(BRL)        int64
property_tax_(BRL)       int64
fire_insurance_(BRL)     int64
total_(BRL)              int64
dtype: object

First few rows:


,city,area,rooms,bathroom,parking_spaces,floor,animal,furniture,hoa_(BRL),rent_amount_(BRL),property_tax_(BRL),fire_insurance_(BRL),total_(BRL)
0,Sao Paulo,70,2,1,1,7,acept,furnished,2065,3300,211,42,5618
1,Sao Paulo,320,4,4,0,20,acept,not furnished,1200,4960,1750,63,7973
2,Porto Alegre,80,1,1,1,6,acept,not furnished,1000,2800,0,41,3841
3,Porto Alegre,51,2,1,0,2,acept,not furnished,270,1112,22,17,1421
4,Sao Paulo,25,1,1,0,1,not acept,not furnished,0,800,25,11,836


In [20]:
brazilian_target = "rent_amount_(BRL)"  # Target variable to impute
brazilian_predictors = ["rooms", "floor", "hoa_(BRL)", "property_tax_(BRL)", "fire_insurance_(BRL)"]

print(f"Target variable: {brazilian_target}")
print(f"Predictors ({len(brazilian_predictors)}): {brazilian_predictors}")

Target variable: rent_amount_(BRL)
Predictors (5): ['rooms', 'floor', 'hoa_(BRL)', 'property_tax_(BRL)', 'fire_insurance_(BRL)']


In [21]:
print("Computing predictor correlations...")
brazilian_correlations = compute_predictor_correlations(
    data=df_brazilian,
    predictors=brazilian_predictors,
    imputed_variables=[brazilian_target],
    method="all"
)

print("\n--- Pearson Correlation Matrix ---")
display(brazilian_correlations["pearson"])

print("\n--- Spearman Correlation Matrix ---")
display(brazilian_correlations["spearman"])

print("\n--- Mutual Information Matrix ---")
display(brazilian_correlations["mutual_info"])

print("\n--- Predictor-Target Mutual Information ---")
display(brazilian_correlations["predictor_target_mi"].sort_values(
    by=brazilian_target, ascending=False
))

Computing predictor correlations...

--- Pearson Correlation Matrix ---


,rooms,floor,hoa_(BRL),property_tax_(BRL),fire_insurance_(BRL)
rooms,1.000000,-0.078687,0.007139,0.075252,0.565148
floor,-0.078687,1.000000,0.019872,0.012626,0.013652
hoa_(BRL),0.007139,0.019872,1.000000,0.007627,0.029535
property_tax_(BRL),0.075252,0.012626,0.007627,1.000000,0.105661
fire_insurance_(BRL),0.565148,0.013652,0.029535,0.105661,1.000000



--- Spearman Correlation Matrix ---


,rooms,floor,hoa_(BRL),property_tax_(BRL),fire_insurance_(BRL)
rooms,1.000000,-0.117345,0.206182,0.595516,0.613788
floor,-0.117345,1.000000,0.618250,0.048770,-0.004572
hoa_(BRL),0.206182,0.618250,1.000000,0.392537,0.293228
property_tax_(BRL),0.595516,0.048770,0.392537,1.000000,0.656049
fire_insurance_(BRL),0.613788,-0.004572,0.293228,0.656049,1.000000



--- Mutual Information Matrix ---


,rooms,floor,hoa_(BRL),property_tax_(BRL),fire_insurance_(BRL)
rooms,1.000000,0.019668,0.102992,0.118588,0.119550
floor,0.019668,1.000000,0.107208,0.014203,0.039879
hoa_(BRL),0.102992,0.107208,1.000000,0.065145,0.066869
property_tax_(BRL),0.118588,0.014203,0.065145,1.000000,0.063229
fire_insurance_(BRL),0.119550,0.039879,0.066869,0.063229,1.000000



--- Predictor-Target Mutual Information ---


,rent_amount_(BRL)
fire_insurance_(BRL),0.412287
property_tax_(BRL),0.057195
hoa_(BRL),0.051921
rooms,0.031211
floor,0.009949


In [22]:
print("Running leave-one-out analysis...")
brazilian_loo = leave_one_out_analysis(
    data=df_brazilian,
    predictors=brazilian_predictors,
    imputed_variables=[brazilian_target],
    model_class=QRF,
    train_size=0.6,
    n_jobs=1
)

print("\n--- Leave-One-Out Results ---")
display(brazilian_loo)

Running leave-one-out analysis...


Leave-one-out analysis:   0%|          | 0/5 [00:00<?, ?it/s]


--- Leave-One-Out Results ---


,predictor_removed,avg_quantile_loss,avg_log_loss,loss_increase,relative_impact,baseline_quantile_loss,baseline_log_loss
4,fire_insurance_(BRL),662.949094,0,611.600644,1191.079089,51.348449,0
1,floor,56.874794,0,5.526345,10.762437,51.348449,0
2,hoa_(BRL),53.955997,0,2.607547,5.078142,51.348449,0
3,property_tax_(BRL),52.573124,0,1.224674,2.385027,51.348449,0
0,rooms,52.341734,0,0.993285,1.934401,51.348449,0


In [23]:
print("Creating 60/40 donor/receiver split...")
brazilian_donor, brazilian_receiver = train_test_split(
    df_brazilian, train_size=0.6, random_state=42
)

brazilian_receiver_no_target = brazilian_receiver.drop(columns=[brazilian_target])

print(f"Donor size: {len(brazilian_donor)}")
print(f"Receiver size: {len(brazilian_receiver)}")

print("\nRunning autoimpute...")
brazilian_result = autoimpute(
    donor_data=brazilian_donor,
    receiver_data=brazilian_receiver_no_target.copy(),
    predictors=brazilian_predictors,
    imputed_variables=[brazilian_target],
    impute_all=True,
    log_level="INFO"
)

print("\n--- Autoimpute CV Results ---")
comparison_viz = method_comparison_results(
    data=brazilian_result.cv_results,
    metric="quantile_loss",
)
fig = comparison_viz.plot(
    title="Brazilian Houses Dataset Benchmarking Results",
    show_mean=True,
)
fig.show()

Creating 60/40 donor/receiver split...
Donor size: 6415
Receiver size: 4277

Running autoimpute...


AutoImputation progress:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   17.8s remaining:   26.7s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   17.8s remaining:   11.9s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   18.0s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.18162083625793457s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    0.2s remaining:    0.3s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    0.2s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    2.7s remaining:    4.1s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    3.2s remaining:    2.1s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | ela


--- Autoimpute CV Results ---


In [24]:
brazilian_result.cv_results.pop('MDN', None)

print("\n--- Autoimpute CV Results (Zoomed In Without MDN) ---")
comparison_viz = method_comparison_results(
    data=brazilian_result.cv_results,
    metric="quantile_loss",
)
fig = comparison_viz.plot(
    title="Brazilian Houses Dataset Benchmarking Results",
    show_mean=True,
)
fig.show()


--- Autoimpute CV Results (Zoomed In Without MDN) ---


In [25]:
print("\n--- Distribution Comparison: Imputed vs Ground Truth (Brazilian Houses) ---")
# Store Wasserstein distances for all methods
brazilian_wasserstein = {}

distribution_comparison_brazilian = compare_distributions(
    donor_data=brazilian_receiver,  # Ground truth test split
    receiver_data=brazilian_result.receiver_data,  # Imputed values
    imputed_variables=[brazilian_target],
)
best_method_name = brazilian_result.fitted_models["best_method"].__class__.__name__ 
best_method_name = best_method_name.replace("Results", "")
brazilian_wasserstein[best_method_name] = distribution_comparison_brazilian[
    distribution_comparison_brazilian['Metric'] == 'wasserstein_distance'
]['Distance'].values[0]
print(f"{best_method_name}: Wasserstein Distance = {brazilian_wasserstein.get(best_method_name, 'N/A'):.4f}")
            
for method_name, imputations in brazilian_result.imputations.items():
    # Skip the 'best_method' (it's a duplicate)
    if method_name == 'best_method':              
        continue
                                                                            
    # Create a copy of receiver data with this method's imputations 
    receiver_with_imputations = brazilian_receiver_no_target.copy() 
                                                    
    # Handle both dict (quantile->DataFrame) and DataFrame formats 
    if isinstance(imputations, dict):                   
        # Get median quantile (0.5) imputations                   
        imp_df = imputations.get(0.5, list(imputations.values())[0]) 
    else:                                                   
        imp_df = imputations
                                   
    # Add imputed values                                    
    for var in [brazilian_target]:
        if var in imp_df.columns:                                    
            receiver_with_imputations[var] = imp_df[var].values    
                                                
    # Calculate distribution comparison                 
    dist_comparison = compare_distributions( 
        donor_data=brazilian_receiver,  # Ground truth                       
        receiver_data=receiver_with_imputations,
        imputed_variables=[brazilian_target],                       
    )

    # Extract Wasserstein distance 
    wd = dist_comparison[dist_comparison['Metric'] == 'wasserstein_distance']['Distance'].values                  
    brazilian_wasserstein[method_name] = wd[0]
                                
    print(f"{method_name}: Wasserstein Distance = {brazilian_wasserstein.get(method_name, 'N/A'):.4f}")
                                                
# Display full comparison for best method
display(distribution_comparison_brazilian)


--- Distribution Comparison: Imputed vs Ground Truth (Brazilian Houses) ---
QRF: Wasserstein Distance = 27.1747
OLS: Wasserstein Distance = 99.3468
QuantReg: Wasserstein Distance = 112.3643
Matching: Wasserstein Distance = 120.4604
MDN: Wasserstein Distance = 3378.6172


,Variable,Metric,Distance
0,rent_amount_(BRL),wasserstein_distance,27.174655


### CIA Sensitivity Analysis

Measuring sensitivity to the Conditional Independence Assumption by progressively removing predictors.

In [26]:
print("Running CIA sensitivity analysis...")
brazilian_cia_results = run_cia_analysis_for_dataset(
    data=df_brazilian,
    predictors=brazilian_predictors,
    imputed_variables=[brazilian_target],
    ordering="mutual_info",
)

# Plot degradation curves
brazilian_cia_fig = plot_cia_degradation_curves(
    brazilian_cia_results,
    title="CIA Sensitivity: Brazilian Houses"
)
brazilian_cia_fig.show()

# Display sensitivity scores
print("\nSensitivity Scores (higher = more sensitive to predictor removal):")
for method, results in brazilian_cia_results.items():
    print(f"  {method}: {results['sensitivity_score']:.3f}")

Running CIA sensitivity analysis...


Progressive exclusion:   0%|          | 0/5 [00:00<?, ?it/s]

  QRF sensitivity score: 14.692


Progressive exclusion:   0%|          | 0/5 [00:00<?, ?it/s]

  OLS sensitivity score: 5.602


...

Progressive exclusion:   0%|          | 0/5 [00:00<?, ?it/s]

  Matching sensitivity score: 3.060


Progressive exclusion:   0%|          | 0/5 [00:00<?, ?it/s]

  MDN sensitivity score: 1.004



Sensitivity Scores (higher = more sensitive to predictor removal):
  QRF: 14.692
  OLS: 5.602
  QuantReg: 7.268
  Matching: 3.060
  MDN: 1.004


---
## 4. Online News Popularity Dataset

News article features and social media shares.

In [27]:
df_news = load_dataset("onlinenewspopularity")
print(f"\nShape: {df_news.shape}")
print(f"\nColumn names (first 30 and last 10):")
cols = list(df_news.columns)
for i, col in enumerate(cols[:30]):
    print(f"  {i+1}. {col}")
print("  ...")
for i, col in enumerate(cols[-10:]):
    print(f"  {len(cols)-10+i+1}. {col}")
print(f"\nData types (sample):")
print(df_news.dtypes[:10])
print(f"\nFirst few rows:")
df_news.head()


Shape: (39644, 60)

Column names (first 30 and last 10):
  1. timedelta
  2. n_tokens_title
  3. n_tokens_content
  4. n_unique_tokens
  5. n_non_stop_words
  6. n_non_stop_unique_tokens
  7. num_hrefs
  8. num_self_hrefs
  9. num_imgs
  10. num_videos
  11. average_token_length
  12. num_keywords
  13. data_channel_is_lifestyle
  14. data_channel_is_entertainment
  15. data_channel_is_bus
  16. data_channel_is_socmed
  17. data_channel_is_tech
  18. data_channel_is_world
  19. kw_min_min
  20. kw_max_min
  21. kw_avg_min
  22. kw_min_max
  23. kw_max_max
  24. kw_avg_max
  25. kw_min_avg
  26. kw_max_avg
  27. kw_avg_avg
  28. self_reference_min_shares
  29. self_reference_max_shares
  30. self_reference_avg_sharess
  ...
  51. min_positive_polarity
  52. max_positive_polarity
  53. avg_negative_polarity
  54. min_negative_polarity
  55. max_negative_polarity
  56. title_subjectivity
  57. title_sentiment_polarity
  58. abs_title_subjectivity
  59. abs_title_sentiment_polarity
  60. 

,timedelta,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,num_videos,average_token_length,num_keywords,data_channel_is_lifestyle,data_channel_is_entertainment,data_channel_is_bus,data_channel_is_socmed,data_channel_is_tech,data_channel_is_world,kw_min_min,kw_max_min,kw_avg_min,kw_min_max,kw_max_max,kw_avg_max,kw_min_avg,kw_max_avg,kw_avg_avg,self_reference_min_shares,self_reference_max_shares,self_reference_avg_sharess,weekday_is_monday,weekday_is_tuesday,weekday_is_wednesday,weekday_is_thursday,weekday_is_friday,weekday_is_saturday,weekday_is_sunday,is_weekend,LDA_00,LDA_01,LDA_02,LDA_03,LDA_04,global_subjectivity,global_sentiment_polarity,global_rate_positive_words,global_rate_negative_words,rate_positive_words,rate_negative_words,avg_positive_polarity,min_positive_polarity,max_positive_polarity,avg_negative_polarity,min_negative_polarity,max_negative_polarity,title_subjectivity,title_sentiment_polarity,abs_title_subjectivity,abs_title_sentiment_polarity,shares
0,731.0,12,219.0,0.663594,1.0,0.815385,4.0,2,1,0,4.680365,5,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,496.0,496.0,496.000000,1,0,0,0,0,0,0,0,0.500331,0.378279,0.040005,0.041263,0.040123,0.521617,0.092562,0.045662,0.013699,0.769231,0.230769,0.378636,0.100000,0.7,-0.350000,-0.600,-0.200000,0.500000,-0.187500,0.000000,0.187500,593.0
1,731.0,9,255.0,0.604743,1.0,0.791946,3.0,1,1,0,4.913725,4,0,0,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1,0,0,0,0,0,0,0,0.799756,0.050047,0.050096,0.050101,0.050001,0.341246,0.148948,0.043137,0.015686,0.733333,0.266667,0.286915,0.033333,0.7,-0.118750,-0.125,-0.100000,0.000000,0.000000,0.500000,0.000000,711.0
2,731.0,9,211.0,0.575130,1.0,0.663866,3.0,1,1,0,4.393365,6,0,0,1,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,918.0,918.0,918.000000,1,0,0,0,0,0,0,0,0.217792,0.033334,0.033351,0.033334,0.682188,0.702222,0.323333,0.056872,0.009479,0.857143,0.142857,0.495833,0.100000,1.0,-0.466667,-0.800,-0.133333,0.000000,0.000000,0.500000,0.000000,1500.0
3,731.0,9,531.0,0.503788,1.0,0.665635,9.0,0,1,0,4.404896,7,0,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1,0,0,0,0,0,0,0,0.028573,0.419300,0.494651,0.028905,0.028572,0.429850,0.100705,0.041431,0.020716,0.666667,0.333333,0.385965,0.136364,0.8,-0.369697,-0.600,-0.166667,0.000000,0.000000,0.500000,0.000000,1200.0
4,731.0,13,1072.0,0.415646,1.0,0.540890,19.0,19,20,0,4.682836,7,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,545.0,16000.0,3151.157895,1,0,0,0,0,0,0,0,0.028633,0.028794,0.028575,0.028572,0.885427,0.513502,0.281003,0.074627,0.012127,0.860215,0.139785,0.411127,0.033333,1.0,-0.220192,-0.500,-0.050000,0.454545,0.136364,0.045455,0.136364,505.0


In [28]:
news_target = "shares"  # Target variable to impute
news_predictors = ["n_tokens_title", "n_tokens_content", "num_hrefs", "num_imgs", "num_videos", "global_subjectivity", "global_sentiment_polarity", "title_subjectivity", "title_sentiment_polarity"]

print(f"Target variable: {news_target}")
print(f"Predictors {len(news_predictors)}: {news_predictors}")

Target variable: shares
Predictors 9: ['n_tokens_title', 'n_tokens_content', 'num_hrefs', 'num_imgs', 'num_videos', 'global_subjectivity', 'global_sentiment_polarity', 'title_subjectivity', 'title_sentiment_polarity']


In [29]:
print("Computing predictor correlations...")
news_correlations = compute_predictor_correlations(
    data=df_news,
    predictors=news_predictors,
    imputed_variables=[news_target],
    method="all"
)

print("\n--- Pearson Correlation Matrix ---")
display(news_correlations["pearson"])

print("\n--- Spearman Correlation Matrix ---")
display(news_correlations["spearman"])

print("\n--- Mutual Information Matrix ---")
display(news_correlations["mutual_info"])

print("\n--- Predictor-Target Mutual Information ---")
display(news_correlations["predictor_target_mi"].sort_values(
    by=news_target, ascending=False
))

Computing predictor correlations...

--- Pearson Correlation Matrix ---


,n_tokens_title,n_tokens_content,num_hrefs,num_imgs,num_videos,global_subjectivity,global_sentiment_polarity,title_subjectivity,title_sentiment_polarity
n_tokens_title,1.000000,0.018160,-0.053496,-0.008858,0.051460,-0.056804,-0.072226,0.077245,0.000240
n_tokens_content,0.018160,1.000000,0.423065,0.342600,0.103699,0.127879,0.021937,0.004484,0.023358
num_hrefs,-0.053496,0.423065,1.000000,0.342633,0.114518,0.203464,0.086859,0.043950,0.039041
num_imgs,-0.008858,0.342600,0.342633,1.000000,-0.067336,0.080468,0.021082,0.056815,0.046310
num_videos,0.051460,0.103699,0.114518,-0.067336,1.000000,0.082052,-0.028434,0.061028,0.021980
global_subjectivity,-0.056804,0.127879,0.203464,0.080468,0.082052,1.000000,0.339436,0.114123,0.034075
global_sentiment_polarity,-0.072226,0.021937,0.086859,0.021082,-0.028434,0.339436,1.000000,0.023620,0.238266
title_subjectivity,0.077245,0.004484,0.043950,0.056815,0.061028,0.114123,0.023620,1.000000,0.232130
title_sentiment_polarity,0.000240,0.023358,0.039041,0.046310,0.021980,0.034075,0.238266,0.232130,1.000000



--- Spearman Correlation Matrix ---


,n_tokens_title,n_tokens_content,num_hrefs,num_imgs,num_videos,global_subjectivity,global_sentiment_polarity,title_subjectivity,title_sentiment_polarity
n_tokens_title,1.000000,0.010237,-0.061430,-0.012670,0.116492,-0.047503,-0.077222,0.108684,0.011174
n_tokens_content,0.010237,1.000000,0.506176,0.391022,-0.037341,0.073055,0.029948,-0.010457,0.010359
num_hrefs,-0.061430,0.506176,1.000000,0.349176,-0.032617,0.193206,0.130599,0.020670,0.026945
num_imgs,-0.012670,0.391022,0.349176,1.000000,-0.168468,0.113552,0.046920,0.023520,0.030852
num_videos,0.116492,-0.037341,-0.032617,-0.168468,1.000000,0.072735,-0.047764,0.059727,0.006032
global_subjectivity,-0.047503,0.073055,0.193206,0.113552,0.072735,1.000000,0.336713,0.134525,0.050840
global_sentiment_polarity,-0.077222,0.029948,0.130599,0.046920,-0.047764,0.336713,1.000000,0.025508,0.222153
title_subjectivity,0.108684,-0.010457,0.020670,0.023520,0.059727,0.134525,0.025508,1.000000,0.351178
title_sentiment_polarity,0.011174,0.010359,0.026945,0.030852,0.006032,0.050840,0.222153,0.351178,1.000000



--- Mutual Information Matrix ---


,n_tokens_title,n_tokens_content,num_hrefs,num_imgs,num_videos,global_subjectivity,global_sentiment_polarity,title_subjectivity,title_sentiment_polarity
n_tokens_title,1.000000,0.000862,0.000354,0.003505,0.003229,0.000254,0.000917,0.008202,0.007850
n_tokens_content,0.000862,1.000000,0.055766,0.050216,0.025189,0.017313,0.015528,0.004697,0.005491
num_hrefs,0.000354,0.055766,1.000000,0.049407,0.022149,0.030201,0.026079,0.003155,0.001383
num_imgs,0.003505,0.050216,0.049407,1.000000,0.044929,0.020942,0.016804,0.006252,0.007037
num_videos,0.003229,0.025189,0.022149,0.044929,1.000000,0.018791,0.016800,0.006123,0.006016
global_subjectivity,0.000254,0.017313,0.030201,0.020942,0.018791,1.000000,0.013351,0.006450,0.004167
global_sentiment_polarity,0.000917,0.015528,0.026079,0.016804,0.016800,0.013351,1.000000,0.004410,0.011626
title_subjectivity,0.008202,0.004697,0.003155,0.006252,0.006123,0.006450,0.004410,1.000000,0.330122
title_sentiment_polarity,0.007850,0.005491,0.001383,0.007037,0.006016,0.004167,0.011626,0.330122,1.000000



--- Predictor-Target Mutual Information ---


,shares
num_imgs,0.002237
global_sentiment_polarity,0.000848
num_hrefs,0.000824
title_subjectivity,0.000505
n_tokens_title,0.000396
title_sentiment_polarity,0.000390
global_subjectivity,0.000360
n_tokens_content,0.000000
num_videos,0.000000


In [30]:
print("Running leave-one-out analysis on top predictors...")
news_loo = leave_one_out_analysis(
    data=df_news,
    predictors=news_predictors,
    imputed_variables=[news_target],
    model_class=QRF,
    train_size=0.6,
    n_jobs=1
)

print("\n--- Leave-One-Out Results ---")
display(news_loo)

Running leave-one-out analysis on top predictors...


Leave-one-out analysis:   0%|          | 0/9 [00:00<?, ?it/s]


--- Leave-One-Out Results ---


,predictor_removed,avg_quantile_loss,avg_log_loss,loss_increase,relative_impact,baseline_quantile_loss,baseline_log_loss
5,global_subjectivity,1446.384028,0,22.472743,1.578240,1423.911286,0
4,num_videos,1445.642897,0,21.731611,1.526191,1423.911286,0
6,global_sentiment_polarity,1436.540498,0,12.629212,0.886938,1423.911286,0
8,title_sentiment_polarity,1427.685541,0,3.774255,0.265063,1423.911286,0
7,title_subjectivity,1423.478042,0,-0.433243,-0.030426,1423.911286,0
0,n_tokens_title,1423.140839,0,-0.770446,-0.054108,1423.911286,0
1,n_tokens_content,1419.151211,0,-4.760075,-0.334296,1423.911286,0
3,num_imgs,1413.187495,0,-10.723790,-0.753122,1423.911286,0
2,num_hrefs,1400.835671,0,-23.075615,-1.620580,1423.911286,0


In [31]:
print("Creating 60/40 donor/receiver split...")

news_donor, news_receiver = train_test_split(
    df_news, train_size=0.6, random_state=42
)

news_receiver_no_target = news_receiver.drop(columns=[news_target])

print(f"Donor size: {len(news_donor)}")
print(f"Receiver size: {len(news_receiver)}")

print("\nRunning autoimpute with top predictors...")
news_result = autoimpute(
    donor_data=news_donor,
    receiver_data=news_receiver_no_target.copy(),
    predictors=news_predictors,
    imputed_variables=[news_target],
    impute_all=True,
    log_level="INFO"
)

print("\n--- Autoimpute CV Results ---")
comparison_viz = method_comparison_results(
    data=news_result.cv_results,
    metric="quantile_loss",
)
fig = comparison_viz.plot(
    title="News Dataset Benchmarking Results",
    show_mean=True,
)
fig.show()

Creating 60/40 donor/receiver split...
Donor size: 23786
Receiver size: 15858

Running autoimpute with top predictors...


AutoImputation progress:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   30.3s remaining:   45.5s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   30.4s remaining:   20.3s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   30.6s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    0.4s remaining:    0.6s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    0.4s remaining:    0.3s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.4s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    5.7s remaining:    8.5s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    6.1s remaining:    4.1s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    6.5s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:   29.2s
[Parallel(n_


--- Autoimpute CV Results ---


In [32]:
print("\n--- Distribution Comparison: Imputed vs Ground Truth (Online News) ---")
# Store Wasserstein distances for all methods
news_wasserstein = {}

distribution_comparison_news = compare_distributions(
    donor_data=news_receiver,  # Ground truth test split
    receiver_data=news_result.receiver_data,  # Imputed values
    imputed_variables=[news_target],
)
best_method_name = news_result.fitted_models["best_method"].__class__.__name__ 
best_method_name = best_method_name.replace("Results", "")
news_wasserstein[best_method_name] = distribution_comparison_news[
    distribution_comparison_news['Metric'] == 'wasserstein_distance'
]['Distance'].values[0]
print(f"{best_method_name}: Wasserstein Distance = {news_wasserstein.get(best_method_name, 'N/A'):.4f}")
            
for method_name, imputations in news_result.imputations.items():
    # Skip the 'best_method' (it's a duplicate)
    if method_name == 'best_method':              
        continue
                                                                            
    # Create a copy of receiver data with this method's imputations 
    receiver_with_imputations = news_receiver_no_target.copy() 
                                                    
    # Handle both dict (quantile->DataFrame) and DataFrame formats 
    if isinstance(imputations, dict):                   
        # Get median quantile (0.5) imputations                   
        imp_df = imputations.get(0.5, list(imputations.values())[0]) 
    else:                                                   
        imp_df = imputations
                                   
    # Add imputed values                                    
    for var in [news_target]:
        if var in imp_df.columns:                                    
            receiver_with_imputations[var] = imp_df[var].values    
                                                
    # Calculate distribution comparison                 
    dist_comparison = compare_distributions( 
        donor_data=news_receiver,  # Ground truth                       
        receiver_data=receiver_with_imputations,
        imputed_variables=[news_target],                       
    )

    # Extract Wasserstein distance 
    wd = dist_comparison[dist_comparison['Metric'] == 'wasserstein_distance']['Distance'].values                  
    news_wasserstein[method_name] = wd[0]
                                
    print(f"{method_name}: Wasserstein Distance = {news_wasserstein.get(method_name, 'N/A'):.4f}")
                                                
# Display full comparison for best method 
display(distribution_comparison_news)


--- Distribution Comparison: Imputed vs Ground Truth (Online News) ---
QuantReg: Wasserstein Distance = 2376.8148
QRF: Wasserstein Distance = 783.7171
OLS: Wasserstein Distance = 2777.1152
Matching: Wasserstein Distance = 239.8596
MDN: Wasserstein Distance = 2038.0328


,Variable,Metric,Distance
0,shares,wasserstein_distance,2376.814836


### CIA Sensitivity Analysis

Measuring sensitivity to the Conditional Independence Assumption by progressively removing predictors.

In [48]:
print("Running CIA sensitivity analysis...")
news_cia_results = run_cia_analysis_for_dataset(
    data=df_news,
    predictors=news_predictors,
    imputed_variables=[news_target],
    ordering="mutual_info",
)

# Plot degradation curves
news_cia_fig = plot_cia_degradation_curves(
    news_cia_results,
    title="CIA Sensitivity: News"
)
news_cia_fig.show()

# Display sensitivity scores
print("\nSensitivity Scores (higher = more sensitive to predictor removal):")
for method, results in news_cia_results.items():
    print(f"  {method}: {results['sensitivity_score']:.3f}")

Running CIA sensitivity analysis...


Progressive exclusion:   0%|          | 0/9 [00:00<?, ?it/s]

  QRF sensitivity score: 1.020


Progressive exclusion:   0%|          | 0/9 [00:00<?, ?it/s]

  OLS sensitivity score: 1.003


...

Progressive exclusion:   0%|          | 0/9 [00:00<?, ?it/s]

  Matching sensitivity score: 1.001


Progressive exclusion:   0%|          | 0/9 [00:00<?, ?it/s]

  MDN sensitivity score: 1.006



Sensitivity Scores (higher = more sensitive to predictor removal):
  QRF: 1.020
  OLS: 1.003
  QuantReg: 1.006
  Matching: 1.001
  MDN: 1.006


---
## 5. Abalone Dataset

Predicting the age of abalone from physical measurements.

In [33]:
df_abalone = load_dataset("abalone")
print(f"\nShape: {df_abalone.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df_abalone.columns):
    print(f"  {i+1}. {col}")
print(f"\nData types:")
print(df_abalone.dtypes)
print(f"\nFirst few rows:")
df_abalone.head()


Shape: (4177, 9)

Column names:
  1. Sex
  2. Length
  3. Diameter
  4. Height
  5. Whole_weight
  6. Shucked_weight
  7. Viscera_weight
  8. Shell_weight
  9. Class_number_of_rings

Data types:
Sex                       object
Length                   float64
Diameter                 float64
Height                   float64
Whole_weight             float64
Shucked_weight           float64
Viscera_weight           float64
Shell_weight             float64
Class_number_of_rings      int64
dtype: object

First few rows:


,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Class_number_of_rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [34]:
abalone_target = "Shell_weight"  # Target variable to impute
abalone_predictors = [col for col in df_abalone.columns if col != abalone_target]

print(f"Target variable: {abalone_target}")
print(f"Predictors ({len(abalone_predictors)}): {abalone_predictors}")

Target variable: Shell_weight
Predictors (8): ['Sex', 'Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Class_number_of_rings']


In [35]:
print("Computing predictor correlations...")
abalone_correlations = compute_predictor_correlations(
    data=df_abalone,
    predictors=abalone_predictors,
    imputed_variables=[abalone_target],
    method="all"
)

print("\n--- Pearson Correlation Matrix ---")
display(abalone_correlations["pearson"])

print("\n--- Spearman Correlation Matrix ---")
display(abalone_correlations["spearman"])

print("\n--- Mutual Information Matrix ---")
display(abalone_correlations["mutual_info"])

print("\n--- Predictor-Target Mutual Information ---")
display(abalone_correlations["predictor_target_mi"].sort_values(
    by=abalone_target, ascending=False
))

Computing predictor correlations...

--- Pearson Correlation Matrix ---


,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Class_number_of_rings
Sex,1.000000,-0.036066,-0.038874,-0.042077,-0.021391,-0.001373,-0.032067,-0.034627
Length,-0.036066,1.000000,0.986812,0.827554,0.925261,0.897914,0.903018,0.556720
Diameter,-0.038874,0.986812,1.000000,0.833684,0.925452,0.893162,0.899724,0.574660
Height,-0.042077,0.827554,0.833684,1.000000,0.819221,0.774972,0.798319,0.557467
Whole_weight,-0.021391,0.925261,0.925452,0.819221,1.000000,0.969405,0.966375,0.540390
Shucked_weight,-0.001373,0.897914,0.893162,0.774972,0.969405,1.000000,0.931961,0.420884
Viscera_weight,-0.032067,0.903018,0.899724,0.798319,0.966375,0.931961,1.000000,0.503819
Class_number_of_rings,-0.034627,0.556720,0.574660,0.557467,0.540390,0.420884,0.503819,1.000000



--- Spearman Correlation Matrix ---


,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Class_number_of_rings
Sex,1.000000,-0.021516,-0.022653,-0.029176,-0.013197,-0.000023,-0.021652,-0.020349
Length,-0.021516,1.000000,0.983319,0.888206,0.972633,0.956830,0.952658,0.604385
Diameter,-0.022653,0.983319,1.000000,0.895705,0.971324,0.950472,0.948391,0.622895
Height,-0.029176,0.888206,0.895705,1.000000,0.915985,0.874196,0.900587,0.657716
Whole_weight,-0.013197,0.972633,0.971324,0.915985,1.000000,0.977060,0.975252,0.630832
Shucked_weight,-0.000023,0.956830,0.950472,0.874196,0.977060,1.000000,0.947635,0.539420
Viscera_weight,-0.021652,0.952658,0.948391,0.900587,0.975252,0.947635,1.000000,0.614344
Class_number_of_rings,-0.020349,0.604385,0.622895,0.657716,0.630832,0.539420,0.614344,1.000000



--- Mutual Information Matrix ---


,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Class_number_of_rings
Sex,1.000000,0.128015,0.118749,0.125884,0.127697,0.125722,0.130254,0.093153
Length,0.128015,1.000000,0.300949,0.183178,0.256054,0.217821,0.204559,0.105884
Diameter,0.118749,0.300949,1.000000,0.188396,0.271122,0.216334,0.206302,0.112585
Height,0.125884,0.183178,0.188396,1.000000,0.207010,0.171274,0.190995,0.101486
Whole_weight,0.127697,0.256054,0.271122,0.207010,1.000000,0.173732,0.174160,0.108976
Shucked_weight,0.125722,0.217821,0.216334,0.171274,0.173732,1.000000,0.134141,0.091902
Viscera_weight,0.130254,0.204559,0.206302,0.190995,0.174160,0.134141,1.000000,0.105824
Class_number_of_rings,0.093153,0.105884,0.112585,0.101486,0.108976,0.091902,0.105824,1.000000



--- Predictor-Target Mutual Information ---


,Shell_weight
Whole_weight,0.186427
Diameter,0.167913
Length,0.159534
Viscera_weight,0.141901
Shucked_weight,0.127999
Height,0.118932
Class_number_of_rings,0.052894
Sex,0.024884


In [36]:
print("Running leave-one-out analysis...")
abalone_loo = leave_one_out_analysis(
    data=df_abalone,
    predictors=abalone_predictors,
    imputed_variables=[abalone_target],
    model_class=QRF,
    train_size=0.6,
    n_jobs=1
)

print("\n--- Leave-One-Out Results ---")
display(abalone_loo)

Running leave-one-out analysis...


Leave-one-out analysis:   0%|          | 0/8 [00:00<?, ?it/s]


--- Leave-One-Out Results ---


,predictor_removed,avg_quantile_loss,avg_log_loss,loss_increase,relative_impact,baseline_quantile_loss,baseline_log_loss
4,Whole_weight,0.011441,0,0.002679,30.572962,0.008762,0
5,Shucked_weight,0.010085,0,0.001322,15.091441,0.008762,0
6,Viscera_weight,0.009000,0,0.000237,2.707903,0.008762,0
3,Height,0.008880,0,0.000118,1.342525,0.008762,0
2,Diameter,0.008771,0,0.000009,0.102489,0.008762,0
0,Sex,0.008755,0,-0.000007,-0.082935,0.008762,0
1,Length,0.008714,0,-0.000048,-0.548617,0.008762,0
7,Class_number_of_rings,0.008650,0,-0.000112,-1.281598,0.008762,0


In [37]:
print("Creating 60/40 donor/receiver split...")
abalone_donor, abalone_receiver = train_test_split(
    df_abalone, train_size=0.6, random_state=42
)

abalone_receiver_no_target = abalone_receiver.drop(columns=[abalone_target])

print(f"Donor size: {len(abalone_donor)}")
print(f"Receiver size: {len(abalone_receiver)}")

print("\nRunning autoimpute...")
abalone_result = autoimpute(
    donor_data=abalone_donor,
    receiver_data=abalone_receiver_no_target.copy(),
    predictors=abalone_predictors,
    imputed_variables=[abalone_target],
    impute_all=True,
    log_level="INFO"
)

print("\n--- Autoimpute CV Results ---")
comparison_viz = method_comparison_results(
    data=abalone_result.cv_results,
    metric="quantile_loss",
)
fig = comparison_viz.plot(
    title="Abalone Dataset Benchmarking Results",
    show_mean=True,
)
fig.show()

Creating 60/40 donor/receiver split...
Donor size: 2506
Receiver size: 1671

Running autoimpute...


AutoImputation progress:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    8.8s remaining:   13.1s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    8.8s remaining:    5.8s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    8.8s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    0.2s remaining:    0.3s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    0.2s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    0.5s remaining:    0.8s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    0.6s remaining:    0.4s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.6s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    3.2s
[Parallel(n_


--- Autoimpute CV Results ---


In [38]:
print("\n--- Distribution Comparison: Imputed vs Ground Truth (Abalone) ---")
# Store Wasserstein distances for all methods
abalone_wasserstein = {}

distribution_comparison_abalone = compare_distributions(
    donor_data=abalone_receiver,  # Ground truth test split
    receiver_data=abalone_result.receiver_data,  # Imputed values
    imputed_variables=[abalone_target],
)
best_method_name = abalone_result.fitted_models["best_method"].__class__.__name__ 
best_method_name = best_method_name.replace("Results", "")
abalone_wasserstein[best_method_name] = distribution_comparison_abalone[
    distribution_comparison_abalone['Metric'] == 'wasserstein_distance'
]['Distance'].values[0]
print(f"{best_method_name}: Wasserstein Distance = {abalone_wasserstein.get(best_method_name, 'N/A'):.4f}")
            
for method_name, imputations in abalone_result.imputations.items():
    # Skip the 'best_method' (it's a duplicate)
    if method_name == 'best_method':              
        continue
                                                                            
    # Create a copy of receiver data with this method's imputations 
    receiver_with_imputations = abalone_receiver_no_target.copy() 
                                                    
    # Handle both dict (quantile->DataFrame) and DataFrame formats 
    if isinstance(imputations, dict):                   
        # Get median quantile (0.5) imputations                   
        imp_df = imputations.get(0.5, list(imputations.values())[0]) 
    else:                                                   
        imp_df = imputations
                                   
    # Add imputed values                                    
    for var in [abalone_target]:
        if var in imp_df.columns:                                    
            receiver_with_imputations[var] = imp_df[var].values    
                                                
    # Calculate distribution comparison                 
    dist_comparison = compare_distributions( 
        donor_data=abalone_receiver,  # Ground truth                       
        receiver_data=receiver_with_imputations,
        imputed_variables=[abalone_target],                       
    )

    # Extract Wasserstein distance 
    wd = dist_comparison[dist_comparison['Metric'] == 'wasserstein_distance']['Distance'].values                  
    abalone_wasserstein[method_name] = wd[0]
                                
    print(f"{method_name}: Wasserstein Distance = {abalone_wasserstein.get(method_name, 'N/A'):.4f}")
                                                
# Display full comparison for best method
display(distribution_comparison_abalone)


--- Distribution Comparison: Imputed vs Ground Truth (Abalone) ---
QuantReg: Wasserstein Distance = 0.0027
QRF: Wasserstein Distance = 0.0028
OLS: Wasserstein Distance = 0.0037
Matching: Wasserstein Distance = 0.0025
MDN: Wasserstein Distance = 0.0404


,Variable,Metric,Distance
0,Shell_weight,wasserstein_distance,0.002713


### CIA Sensitivity Analysis

Measuring sensitivity to the Conditional Independence Assumption by progressively removing predictors.

In [39]:
print("Running CIA sensitivity analysis...")
abalone_cia_results = run_cia_analysis_for_dataset(
    data=df_abalone,
    predictors=abalone_predictors,
    imputed_variables=[abalone_target],
    ordering="mutual_info",
)

# Plot degradation curves
abalone_cia_fig = plot_cia_degradation_curves(
    abalone_cia_results,
    title="CIA Sensitivity: Abalone"
)
abalone_cia_fig.show()

# Display sensitivity scores
print("\nSensitivity Scores (higher = more sensitive to predictor removal):")
for method, results in abalone_cia_results.items():
    print(f"  {method}: {results['sensitivity_score']:.3f}")

Running CIA sensitivity analysis...


Progressive exclusion:   0%|          | 0/8 [00:00<?, ?it/s]

  QRF sensitivity score: 2.071


Progressive exclusion:   0%|          | 0/8 [00:00<?, ?it/s]

  OLS sensitivity score: 2.067


...

Progressive exclusion:   0%|          | 0/8 [00:00<?, ?it/s]

  Matching sensitivity score: 1.886


Progressive exclusion:   0%|          | 0/8 [00:00<?, ?it/s]

  MDN sensitivity score: 1.472



Sensitivity Scores (higher = more sensitive to predictor removal):
  QRF: 2.071
  OLS: 2.067
  QuantReg: 2.115
  Matching: 1.886
  MDN: 1.472


---
## 6. House Sales Dataset

King County house sales data with property features and prices.

In [40]:
df_house = load_dataset("house_sales")
print(f"\nShape: {df_house.shape}")
print(f"\nColumn names:")
for i, col in enumerate(df_house.columns):
    print(f"  {i+1}. {col}")
print(f"\nData types:")
print(df_house.dtypes)
print(f"\nFirst few rows:")
df_house.head()


Shape: (21613, 22)

Column names:
  1. bedrooms
  2. bathrooms
  3. sqft_living
  4. sqft_lot
  5. floors
  6. waterfront
  7. view
  8. condition
  9. grade
  10. sqft_above
  11. sqft_basement
  12. yr_built
  13. yr_renovated
  14. zipcode
  15. lat
  16. long
  17. sqft_living15
  18. sqft_lot15
  19. date_year
  20. date_month
  21. date_day
  22. price

Data types:
bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
yr_built           int64
yr_renovated       int64
zipcode            int64
lat              float64
long             float64
sqft_living15      int64
sqft_lot15         int64
date_year          int64
date_month         int64
date_day           int64
price            float64
dtype: object

First few rows:


,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date_year,date_month,date_day,price
0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650,2014,10,13,221900.0
1,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639,2014,12,9,538000.0
2,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062,2015,2,25,180000.0
3,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000,2014,12,9,604000.0
4,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503,2015,2,18,510000.0


In [41]:
house_target = "price"  # Target variable to impute
house_predictors = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "waterfront", "view", "condition", "grade", "sqft_above", "sqft_basement"]

print(f"Target variable: {house_target}")
print(f"Predictors ({len(house_predictors)}): {house_predictors}")

Target variable: price
Predictors (10): ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement']


In [42]:
print("Computing predictor correlations...")
house_correlations = compute_predictor_correlations(
    data=df_house,
    predictors=house_predictors,
    imputed_variables=[house_target],
    method="all"
)

print("\n--- Pearson Correlation Matrix ---")
display(house_correlations["pearson"])

print("\n--- Spearman Correlation Matrix ---")
display(house_correlations["spearman"])

print("\n--- Mutual Information Matrix ---")
display(house_correlations["mutual_info"])

print("\n--- Predictor-Target Mutual Information ---")
display(house_correlations["predictor_target_mi"].sort_values(
    by=house_target, ascending=False
))

Computing predictor correlations...

--- Pearson Correlation Matrix ---


,bedrooms,bathrooms,sqft_living,sqft_lot,waterfront,view,condition,grade,sqft_above,sqft_basement
bedrooms,1.000000,0.515884,0.576671,0.031703,-0.006582,0.079532,0.028472,0.356967,0.477600,0.303093
bathrooms,0.515884,1.000000,0.754665,0.087740,0.063744,0.187737,-0.124982,0.664983,0.685342,0.283770
sqft_living,0.576671,0.754665,1.000000,0.172826,0.103818,0.284611,-0.058753,0.762704,0.876597,0.435043
sqft_lot,0.031703,0.087740,0.172826,1.000000,0.021604,0.074710,-0.008958,0.113621,0.183512,0.015286
waterfront,-0.006582,0.063744,0.103818,0.021604,1.000000,0.401857,0.016653,0.082775,0.072075,0.080588
view,0.079532,0.187737,0.284611,0.074710,0.401857,1.000000,0.045990,0.251321,0.167649,0.276947
condition,0.028472,-0.124982,-0.058753,-0.008958,0.016653,0.045990,1.000000,-0.144674,-0.158214,0.174105
grade,0.356967,0.664983,0.762704,0.113621,0.082775,0.251321,-0.144674,1.000000,0.755923,0.168392
sqft_above,0.477600,0.685342,0.876597,0.183512,0.072075,0.167649,-0.158214,0.755923,1.000000,-0.051943
sqft_basement,0.303093,0.283770,0.435043,0.015286,0.080588,0.276947,0.174105,0.168392,-0.051943,1.000000



--- Spearman Correlation Matrix ---


,bedrooms,bathrooms,sqft_living,sqft_lot,waterfront,view,condition,grade,sqft_above,sqft_basement
bedrooms,1.000000,0.521406,0.647366,0.216531,-0.007598,0.081421,0.012660,0.380534,0.539704,0.230496
bathrooms,0.521406,1.000000,0.745526,0.068805,0.049522,0.155549,-0.162891,0.658194,0.691006,0.191848
sqft_living,0.647366,0.745526,1.000000,0.304159,0.070326,0.232994,-0.062638,0.716400,0.843504,0.327878
sqft_lot,0.216531,0.068805,0.304159,1.000000,0.085601,0.117033,0.114724,0.152049,0.272408,0.036624
waterfront,-0.007598,0.049522,0.070326,0.085601,1.000000,0.284924,0.016744,0.062189,0.054490,0.051969
view,0.081421,0.155549,0.232994,0.117033,0.284924,1.000000,0.046000,0.217044,0.144394,0.236525
condition,0.012660,-0.162891,-0.062638,0.114724,0.016744,0.046000,1.000000,-0.167374,-0.158126,0.161623
grade,0.380534,0.658194,0.716400,0.152049,0.062189,0.217044,-0.167374,1.000000,0.711815,0.092927
sqft_above,0.539704,0.691006,0.843504,0.272408,0.054490,0.144394,-0.158126,0.711815,1.000000,-0.165644
sqft_basement,0.230496,0.191848,0.327878,0.036624,0.051969,0.236525,0.161623,0.092927,-0.165644,1.000000



--- Mutual Information Matrix ---


,bedrooms,bathrooms,sqft_living,sqft_lot,waterfront,view,condition,grade,sqft_above,sqft_basement
bedrooms,1.000000,0.098199,0.159197,0.038320,0.037347,0.007649,0.006323,0.049392,0.112457,0.034205
bathrooms,0.098199,1.000000,0.151131,0.028646,0.056060,0.029864,0.034050,0.148806,0.117633,0.030131
sqft_living,0.159197,0.151131,1.000000,0.015266,0.049643,0.047129,0.010742,0.200484,0.405167,0.087963
sqft_lot,0.038320,0.028646,0.015266,1.000000,0.081274,0.031370,0.033865,0.043008,0.016837,0.019837
waterfront,0.037347,0.056060,0.049643,0.081274,1.000000,0.418037,0.000000,0.050705,0.028753,0.215099
view,0.007649,0.029864,0.047129,0.031370,0.418037,1.000000,0.000000,0.043477,0.023920,0.074452
condition,0.006323,0.034050,0.010742,0.033865,0.000000,0.000000,1.000000,0.020681,0.020348,0.018175
grade,0.049392,0.148806,0.200484,0.043008,0.050705,0.043477,0.020681,1.000000,0.193197,0.020057
sqft_above,0.112457,0.117633,0.405167,0.016837,0.028753,0.023920,0.020348,0.193197,1.000000,0.043748
sqft_basement,0.034205,0.030131,0.087963,0.019837,0.215099,0.074452,0.018175,0.020057,0.043748,1.000000



--- Predictor-Target Mutual Information ---


,price
sqft_living,0.034919
grade,0.033545
sqft_above,0.025859
bathrooms,0.020110
bedrooms,0.007915
sqft_basement,0.006930
sqft_lot,0.006063
view,0.005654
condition,0.001584
waterfront,0.001340


In [43]:
print("Running leave-one-out analysis...")
house_loo = leave_one_out_analysis(
    data=df_house,
    predictors=house_predictors,
    imputed_variables=[house_target],
    model_class=QRF,
    train_size=0.6,
    n_jobs=1
)

print("\n--- Leave-One-Out Results ---")
display(house_loo)

Running leave-one-out analysis...


Leave-one-out analysis:   0%|          | 0/10 [00:00<?, ?it/s]


--- Leave-One-Out Results ---


,predictor_removed,avg_quantile_loss,avg_log_loss,loss_increase,relative_impact,baseline_quantile_loss,baseline_log_loss
7,grade,69216.847912,0,6095.055599,9.656024,63121.792313,0
3,sqft_lot,68609.389908,0,5487.597595,8.693666,63121.792313,0
6,condition,64704.662524,0,1582.870212,2.507645,63121.792313,0
1,bathrooms,64190.973160,0,1069.180848,1.693838,63121.792313,0
5,view,63788.472535,0,666.680222,1.056181,63121.792313,0
0,bedrooms,63643.886195,0,522.093882,0.827121,63121.792313,0
2,sqft_living,63438.111978,0,316.319666,0.501126,63121.792313,0
4,waterfront,63387.546779,0,265.754467,0.421019,63121.792313,0
9,sqft_basement,63230.876154,0,109.083842,0.172815,63121.792313,0
8,sqft_above,63195.338181,0,73.545868,0.116514,63121.792313,0


In [44]:
print("Creating 60/40 donor/receiver split...")
house_donor, house_receiver = train_test_split(
    df_house, train_size=0.6, random_state=42
)

house_receiver_no_target = house_receiver.drop(columns=[house_target])

print(f"Donor size: {len(house_donor)}")
print(f"Receiver size: {len(house_receiver)}")

print("\nRunning autoimpute...")
house_result = autoimpute(
    donor_data=house_donor,
    receiver_data=house_receiver_no_target.copy(),
    predictors=house_predictors,
    imputed_variables=[house_target],
    impute_all=True,
    log_level="INFO"
)

print("\n--- Autoimpute CV Results ---")
comparison_viz = method_comparison_results(
    data=house_result.cv_results,
    metric="quantile_loss",
)
fig = comparison_viz.plot(
    title="House Sales Dataset Benchmarking Results",
    show_mean=True,
)
fig.show()

Creating 60/40 donor/receiver split...
Donor size: 12967
Receiver size: 8646

Running autoimpute...


AutoImputation progress:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   10.5s remaining:   15.8s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   10.5s remaining:    7.0s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   10.6s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    0.3s remaining:    0.4s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    0.3s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   19.3s remaining:   29.0s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:   20.0s remaining:   13.3s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   20.2s finished
[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:   16.8s
[Parallel(n_


--- Autoimpute CV Results ---


In [45]:
house_result.cv_results.pop('MDN', None)

print("\n--- Autoimpute CV Results ---")
comparison_viz = method_comparison_results(
    data=house_result.cv_results,
    metric="quantile_loss",
)
fig = comparison_viz.plot(
    title="House Sales Dataset Benchmarking Results",
    show_mean=True,
)
fig.show()


--- Autoimpute CV Results ---


In [46]:
print("\n--- Distribution Comparison: Imputed vs Ground Truth (House Sales) ---")
# Store Wasserstein distances for all methods
house_wasserstein = {}

distribution_comparison_house = compare_distributions(
    donor_data=house_receiver,  # Ground truth test split
    receiver_data=house_result.receiver_data,  # Imputed values
    imputed_variables=[house_target],
)
best_method_name = house_result.fitted_models["best_method"].__class__.__name__ 
best_method_name = best_method_name.replace("Results", "")
house_wasserstein[best_method_name] = distribution_comparison_house[
    distribution_comparison_house['Metric'] == 'wasserstein_distance'
]['Distance'].values[0]
print(f"{best_method_name}: Wasserstein Distance = {house_wasserstein.get(best_method_name, 'N/A'):.4f}")
            
for method_name, imputations in house_result.imputations.items():
    # Skip the 'best_method' (it's a duplicate)
    if method_name == 'best_method':              
        continue
                                                                            
    # Create a copy of receiver data with this method's imputations 
    receiver_with_imputations = house_receiver_no_target.copy() 
                                                    
    # Handle both dict (quantile->DataFrame) and DataFrame formats 
    if isinstance(imputations, dict):                   
        # Get median quantile (0.5) imputations                   
        imp_df = imputations.get(0.5, list(imputations.values())[0]) 
    else:                                                   
        imp_df = imputations
                                   
    # Add imputed values                                    
    for var in [house_target]:
        if var in imp_df.columns:                                    
            receiver_with_imputations[var] = imp_df[var].values    
                                                
    # Calculate distribution comparison                 
    dist_comparison = compare_distributions( 
        donor_data=house_receiver,  # Ground truth                       
        receiver_data=receiver_with_imputations,
        imputed_variables=[house_target],                       
    )

    # Extract Wasserstein distance 
    wd = dist_comparison[dist_comparison['Metric'] == 'wasserstein_distance']['Distance'].values                  
    house_wasserstein[method_name] = wd[0]
                                
    print(f"{method_name}: Wasserstein Distance = {house_wasserstein.get(method_name, 'N/A'):.4f}")
                                                
# Display full comparison for best method
display(distribution_comparison_house)


--- Distribution Comparison: Imputed vs Ground Truth (House Sales) ---
QuantReg: Wasserstein Distance = 64725.2760
QRF: Wasserstein Distance = 12576.6113
OLS: Wasserstein Distance = 41862.2873
Matching: Wasserstein Distance = 8797.8007
MDN: Wasserstein Distance = 544730.3917


,Variable,Metric,Distance
0,price,wasserstein_distance,64725.276019


### CIA Sensitivity Analysis

Measuring sensitivity to the Conditional Independence Assumption by progressively removing predictors.

In [47]:
print("Running CIA sensitivity analysis...")
house_cia_results = run_cia_analysis_for_dataset(
    data=df_house,
    predictors=house_predictors,
    imputed_variables=[house_target],
    ordering="mutual_info",
)

# Plot degradation curves
house_cia_fig = plot_cia_degradation_curves(
    house_cia_results,
    title="CIA Sensitivity: House"
)
house_cia_fig.show()

# Display sensitivity scores
print("\nSensitivity Scores (higher = more sensitive to predictor removal):")
for method, results in house_cia_results.items():
    print(f"  {method}: {results['sensitivity_score']:.3f}")

Running CIA sensitivity analysis...


Progressive exclusion:   0%|          | 0/10 [00:00<?, ?it/s]

  QRF sensitivity score: 1.460


Progressive exclusion:   0%|          | 0/10 [00:00<?, ?it/s]

  OLS sensitivity score: 1.298


...

Progressive exclusion:   0%|          | 0/10 [00:00<?, ?it/s]

  Matching sensitivity score: 1.277


Progressive exclusion:   0%|          | 0/10 [00:00<?, ?it/s]

  MDN sensitivity score: 0.982



Sensitivity Scores (higher = more sensitive to predictor removal):
  QRF: 1.460
  OLS: 1.298
  QuantReg: 1.326
  Matching: 1.277
  MDN: 0.982


---
## Cross-Dataset Results Summary

Aggregating results across all benchmarking datasets.

In [52]:
# Collect all CV results
all_cv_results = {
    "space_ga": space_ga_result.cv_results,
    "elevators": elevators_result.cv_results,
    "brazilian_houses": brazilian_result.cv_results,
    "onlinenewspopularity": news_result.cv_results,
    "abalone": abalone_result.cv_results,
    "house_sales": house_result.cv_results,
}
all_cv_results_df = pd.DataFrame(all_cv_results).T
all_cv_results_df.to_csv("benchmark_cv_results.csv")

# Collect all Wasserstein results            
all_wasserstein_results = {
    "space_ga": space_ga_wasserstein,       
    "elevators": elevators_wasserstein, 
    "brazilian_houses": brazilian_wasserstein,             
    "onlinenewspopularity": news_wasserstein, 
    "abalone": abalone_wasserstein,          
    "house_sales": house_wasserstein,      
} 
all_wasserstein_results_df = pd.DataFrame(all_wasserstein_results).T
all_wasserstein_results_df.to_csv("benchmark_wasserstein_results.csv")

# Collect all CIA results
all_cia_results = {
    "space_ga": space_ga_cia_results,
    "elevators": elevators_cia_results,
    "brazilian_houses": brazilian_cia_results,
    "onlinenewspopularity": news_cia_results,
    "abalone": abalone_cia_results,
    "house_sales": house_cia_results,
}
all_cia_results_df = pd.DataFrame(all_cia_results).T
all_cia_results_df.to_csv("benchmark_cia_results.csv")

# Create summary table
print("\n=== Cross-Dataset Benchmark Summary Table ===\n")
summary_table = create_benchmark_summary_table(all_cv_results, all_cia_results)
display(summary_table)


=== Cross-Dataset Benchmark Summary Table ===



,Dataset,Best Method,QRF Rank,QRF Loss,OLS Rank,OLS Loss,QuantReg Rank,QuantReg Loss,Matching Rank,Matching Loss,MDN Rank,MDN Loss,QRF CIA,OLS CIA,QuantReg CIA,Matching CIA,MDN CIA
0,space_ga,OLS,2.0,0.038926,1.000000,0.036167,5.000000,0.084055,4.000000,0.062758,3.000000,0.050411,1.442547,1.191864,0.687691,1.267156,1.312439
1,elevators,QuantReg,3.0,0.001321,2.000000,0.001283,1.000000,0.001181,4.000000,0.002634,5.000000,NaN,1.292881,1.180257,1.173917,1.050390,2.055043
2,brazilian_houses,QRF,1.0,52.164667,3.000000,126.926767,2.000000,104.936007,4.000000,398.728371,5.000000,NaN,14.691979,5.601620,7.267652,3.059647,1.004445
3,onlinenewspopularity,QuantReg,3.0,1357.210240,4.000000,1800.669662,1.000000,1045.319334,5.000000,1984.738231,2.000000,1227.693208,1.019769,1.002792,1.006457,1.001464,1.006107
4,abalone,QuantReg,3.0,0.009056,2.000000,0.007400,1.000000,0.006479,4.000000,0.015145,5.000000,0.025759,2.071249,2.066912,2.115189,1.885606,1.471810
5,house_sales,QuantReg,3.0,60952.489182,2.000000,58409.524276,1.000000,54897.719026,4.000000,102881.830903,5.000000,NaN,1.459985,1.297931,1.326381,1.277018,0.982165
6,Mean Rank,-,2.5,10393.652232,2.333333,10056.194259,1.833333,9341.344347,4.166667,17544.229674,4.166667,409.256459,3.663068,2.056896,2.262881,1.590214,1.305335


In [2]:
all_cv_results = pd.read_csv("benchmark_cv_results.csv", index_col=0).to_dict()
all_wasserstein_results = pd.read_csv("benchmark_wasserstein_results.csv", index_col=0).to_dict()
all_cia_results = pd.read_csv("benchmark_cia_results.csv", index_col=0).to_dict()

In [4]:
# Create benchmark heatmap
print("\n=== Cross-Dataset Benchmark Heatmap ===\n")
heatmap_fig = create_benchmark_heatmap(
    all_cv_results,
    wasserstein_results=all_wasserstein_results,
    title="Cross-Dataset Performance Comparison"
)
heatmap_fig.show()


=== Cross-Dataset Benchmark Heatmap ===



---
## Summary

This notebook has analyzed 6 benchmarking datasets:

1. **space_ga** - Georgia voting data (3,107 rows, 7 cols)
2. **elevators** - Aircraft control data (16,599 rows, 19 cols)
3. **brazilian_houses** - Real estate data (10,692 rows, 13 cols)
4. **onlinenewspopularity** - News article data (39,644 rows, 60 cols)
5. **abalone** - Marine biology data (4,177 rows, 9 cols)
6. **house_sales** - King County real estate (21,613 rows, 22 cols)

For each dataset:
- Computed predictor correlations (Pearson, Spearman, Mutual Information)
- Performed leave-one-out analysis to identify important predictors
- Split data into donor/receiver sets (60/40)
- Ran autoimpute to compare QRF, OLS, QuantReg, Matching, and MDN models
- Compared imputed vs ground truth distributions